In [23]:
import seaborn as sns
import pandas as pd
import numpy as np
from sklearn.datasets import make_classification # used to generate a random sample for CLASSIFICATION problems
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

---
## <u>Generate Dataset</u>

In [17]:
X, y = make_classification(   # no encoding or scaling need (scaling only needed if we make a penalization model like L1/L2)
    n_samples = 10000,    
    n_features = 15,              
    n_informative = 12,
    n_classes = 2,            # binary classification, we can do multiclass as well
    random_state = 42
)

---
## <u>Train Test Split</u>

In [18]:
X = pd.DataFrame(X)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

X_train.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
9254,3.223624,0.861537,2.381547,1.395780,4.014657,-1.527077,3.048176,1.264915,3.135539,3.286052,1.067258,-5.254813,-1.072045,-1.154300,-2.089999
1561,-3.577916,0.271841,-2.039949,4.737522,2.042567,2.968547,3.149826,4.125000,2.379183,0.855071,6.071050,7.298252,-6.449665,0.570726,-1.825491
1670,-3.274038,-1.223501,-2.915686,1.914367,-1.074814,0.016904,-1.189167,-1.681084,-0.641283,1.205698,-4.650563,-1.960073,4.951817,1.183464,-2.243373
6087,0.850901,-0.702375,-0.947497,2.046136,2.498039,1.364085,3.353895,-2.033510,2.892577,2.547747,2.607806,0.766182,-1.234793,0.483933,-1.407348
6669,0.471267,-0.092617,-2.245596,-1.257869,2.061824,-2.211428,0.891296,1.083817,-1.848658,0.857356,-1.442283,-0.577073,1.419910,0.391048,1.179483


---
# <u>Create model, Train and Predict</u>

In [19]:
# making the pipeline to do hyperparamter tuning also

full_tree = DecisionTreeClassifier(random_state = 42)
full_tree.fit(X_train, y_train)

path = full_tree.cost_complexity_pruning_path(X_train, y_train)
ccp_alphas = path.ccp_alphas

reduced_ccp_alphas = ccp_alphas[::30] # shape = (13,)

steps = [("GBC", GradientBoostingClassifier(random_state = 42))]
pipeline = Pipeline(steps)

param_grid = {
    "GBC__n_estimators" : [100, 200, 300],         
    "GBC__learning_rate" : [0.01, 0.05, 0.1, 0.2],     
    "GBC__max_depth" : [2, 3, 4, 5],
    "GBC__subsample" : [0.7, 0.8, 0.9, 1.0],        # the fraction of samples to be used for fitting individual weak learners (DT)     
    "GBC__ccp_alpha" : reduced_ccp_alphas   
}

GBR_cv = RandomizedSearchCV(
    pipeline,
    param_grid,
    cv = 5,
    n_iter = 20,
    n_jobs = -1,
    random_state = 42
)

GBR_cv.fit(X_train, y_train)
y_training_pred = GBR_cv.predict(X_train)
y_test_pred = GBR_cv.predict(X_test)

---
# <u>Evaluate</u>

In [26]:
print("For Gradient Boosting Classifier (hyperparameter tuning) :-\n")

print("Best Parameters : ", GBR_cv.best_params_)

print("\nTraining scores :-")
print("Train Accuracy : ", accuracy_score(y_train, y_training_pred))
print("Train precision : ", precision_score(y_train, y_training_pred))
print("Train recall : ", recall_score(y_train, y_training_pred))
print("Train F1  : ", f1_score(y_train, y_training_pred))
print("Train confusion matrix : \n", confusion_matrix(y_train, y_training_pred))


print("\nTesting scores :-")
print("\nTest Accuracy : ", accuracy_score(y_test, y_test_pred))
print("\nTest precision : ", precision_score(y_test, y_test_pred))
print("\nTest recall : ", recall_score(y_test, y_test_pred))
print("\nTest F1 : ", f1_score(y_test, y_test_pred))
print("\nTest confusion matrix : \n", confusion_matrix(y_test, y_test_pred))

# result, this model is a bit overfitting

For Gradient Boosting Classifier (hyperparameter tuning) :-

Best Parameters :  {'GBC__subsample': 0.9, 'GBC__n_estimators': 300, 'GBC__max_depth': 4, 'GBC__learning_rate': 0.1, 'GBC__ccp_alpha': np.float64(0.0)}

Training scores :-
Train Accuracy :  0.996125
Train precision :  0.9953086419753087
Train recall :  0.9970319069997526
Train F1  :  0.9961695292227851
Train confusion matrix : 
 [[3938   19]
 [  12 4031]]

Testing scores :-

Test Accuracy :  0.9535

Test precision :  0.9424460431654677

Test recall :  0.9612159329140462

Test F1 :  0.9517384535547483

Test confusion matrix : 
 [[990  56]
 [ 37 917]]
